# Architektura Aplikacji w Pythonie — Zestaw Zaliczeniowy

**WSEI Kraków · semestr letni 2026 · prowadzący: Michał Madejski**

---

## Filozofia tego zestawu

Sześć laboratoriów dało Ci sześć narzędzi. Ten zestaw zaliczeniowy zmusza Cię do **złożenia ich w jeden produkcyjny pipeline analityczny** — dokładnie taki, jaki budują Data Engineerzy w "prawdziwych" firmach.

**Wspólny dataset:** [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) z Hugging Face Hub — 50 000 recenzji filmów z etykietami sentymentu (pozytywna / negatywna).

**Reguły:**
1. Każdy lab ma blok: **Teoria → Przykład rozwiązany → Zadanie samodzielne**.
2. Zadania samodzielne **rozszerzają** przykład — dokładnie ten sam pattern, inny scenariusz.
3. Cały notebook ma być **uruchamialny od góry do dołu**. Brak hardkodowanych ścieżek, brak ręcznych downloadów.
4. Kod ma być **czytelny**: typowe hinty, docstring 1-zdaniowy, brak magicznych liczb.

**Ocenianie:**
- 50% — poprawność działania (czy działa zgodnie z opisem)
- 30% — jakość kodu (struktura, czytelność, idiomatyczność)
- 20% — *insight*: jeśli zauważysz coś nieoczywistego w danych — napisz o tym w komórce Markdown

---

## Mapa zestawu

| # | Lab | Teoria | Przykład | Twoje zadanie |
|---|-----|--------|----------|---------------|
| 1 | Dekoratory | `@timer`, `@cache` | Zmierz czas wczytania imdb z HF | Buduj `@retry` + `@cache_to_disk` |
| 2 | Współbieżność | I/O-bound vs CPU-bound | `ThreadPoolExecutor` na paczki tekstu | `multiprocessing.Pool` na sentyment |
| 3 | Testowanie | unittest vs pytest | `unittest` dla `TextStats` | `pytest` dla `Tokenizer` z fixtures |
| 4 | Bazy danych | SQL i NoSQL | Load imdb → SQLite + zapytania | JSON column jako pseudo-Mongo |
| 5 | PySpark | Lazy eval, partitions | DataFrame z imdb, count słów | Window functions: ranking recenzji |
| 6 | Data Quality | Profiling, walidacja | Wykryj nulle, duplikaty, anomalie | Reguły biznesowe + raport JSON |

---

## Setup

In [1]:
# Globalna konfiguracja -- jedna komorka, jeden raz
import os, sys, time, json, warnings, random, textwrap
from pathlib import Path
warnings.filterwarnings("ignore")

WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

# Tame log spamu HF Datasets
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BAR", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# Na Windowsie czasem psuja sie certyfikaty systemowe.
# Certifi daje zwykly plik CA, ktory dziala stabilniej.
try:
    import ssl
    import certifi

    if not getattr(ssl.create_default_context, "_uses_certifi", False):
        _old_create_default_context = ssl.create_default_context

        def _create_default_context(purpose=ssl.Purpose.SERVER_AUTH, cafile=None, capath=None, cadata=None):
            if cafile is None and capath is None and cadata is None:
                cafile = certifi.where()
            return _old_create_default_context(purpose=purpose, cafile=cafile, capath=capath, cadata=cadata)

        _create_default_context._uses_certifi = True
        ssl.create_default_context = _create_default_context

    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception as e:
    print(f"Nie udalo sie ustawic certyfikatow SSL: {e}")

print(f"Python: {sys.version.split()[0]}")
print(f"Workspace: {WORKDIR.resolve()}")


Python: 3.11.15
Workspace: C:\Users\jakub\Documents\Codex\2026-06-26\s\outputs\zestaw_zaliczeniowy\_workspace


---

# Lab 1 — Dekoratory

## Teoria w trzech zdaniach

**Dekorator** to funkcja, która przyjmuje funkcję i zwraca funkcję. Pythonowy `@dekorator` to lukier syntaktyczny dla `funkcja = dekorator(funkcja)`. Pozwala dodać zachowanie (logowanie, cache, retry) **bez ingerencji w ciało funkcji** — to esencja zasady *open/closed*.

### Wzorzec dekoratora z argumentami

```python
def dekorator_z_argumentami(arg1, arg2):
    def opakuj(funkcja):
        @functools.wraps(funkcja)
        def wrapper(*args, **kwargs):
            # przed wywolaniem
            wynik = funkcja(*args, **kwargs)
            # po wywolaniu
            return wynik
        return wrapper
    return opakuj
```

Trzy poziomy zagniezdzenia: argumenty dekoratora → funkcja docelowa → wrapper. **Zapamiętaj ten układ raz — reszta to wariacje.**

## Przykład rozwiązany: `@timer` + `@cache` na ładowaniu z Hugging Face

In [2]:
import functools
from datasets import load_dataset

def timer(func):
    """Mierzy czas wykonania funkcji i drukuje wynik."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - t0
        print(f"  [timer] {func.__name__} -> {elapsed:.2f}s")
        return result
    return wrapper

@timer
@functools.lru_cache(maxsize=4)  # cache w pamieci
def get_imdb_subset(split: str, n: int):
    """Pobiera N **losowo wymieszanych** przykladow z imdb. Cachuje wynik w RAM.

    UWAGA: dataset stanfordnlp/imdb jest na HF zsortowany po labelu
    (0..12499 = neg, 12500..24999 = pos). Bez shuffle dostalibysmy
    100% jednej klasy dla N <= 12500. .shuffle(seed=42) gwarantuje
    rownomierna probke.
    """
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

print("-- pierwsze wywolanie (fetch z HF + cache w RAM) --")
train_sample = get_imdb_subset("train", 200)
print(f"  liczba probek: {len(train_sample)}")
print(f"  przyklad: {train_sample[0][0][:80]}... -> label={train_sample[0][1]}")

# Sanity check: czy mamy obie klasy?
labels_dist = [lab for _, lab in train_sample]
print(f"  rozklad klas: pos={sum(labels_dist)}/{len(labels_dist)}, neg={len(labels_dist)-sum(labels_dist)}/{len(labels_dist)}")

print("\n-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --")
_ = get_imdb_subset("train", 200)

-- pierwsze wywolanie (fetch z HF + cache w RAM) --


  [timer] get_imdb_subset -> 1.88s
  liczba probek: 200
  przyklad: There is no relation at all between Fortier and Profiler but the fact that both ... -> label=1
  rozklad klas: pos=96/200, neg=104/200

-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --
  [timer] get_imdb_subset -> 0.00s


## Zadanie 1.1 — `@retry` + `@cache_to_disk`

**Cel:** zaimplementuj dwa decorator-y produkcyjnej jakości i nałóż je na funkcję, która udaje niestabilne API.

**Wymagania:**

1. `@retry(max_attempts: int, delay: float, backoff: float = 2.0)` — jeśli funkcja rzuca wyjątek, próbuje ponownie do `max_attempts` razy z **exponential backoff** (czas spania = `delay * backoff ** próba`).
2. `@cache_to_disk(cache_dir: Path)` — zapisuje wynik do pliku JSON w `cache_dir`. Klucz cache to hash argumentów. Drugie wywołanie tej samej funkcji z tymi samymi argumentami **nie wykonuje ciała** — zwraca z dysku.
3. Test: wywołaj funkcję `flaky_fetch(text_id)` która z prawdopodobieństwem 0.5 rzuca `ValueError`. Powinna **prawie zawsze** się udać dzięki retry. Drugie wywołanie z tym samym `text_id` powinno trafić w cache.

**Insight do raportu:** jak zmienia się szansa sukcesu wraz z `max_attempts`? Policz to teoretycznie (P(sukces) = 1 - 0.5^N) i porównaj z eksperymentem na 100 wywołaniach.

In [3]:
import hashlib

# TODO Zadanie 1.1: zaimplementuj dwa dekoratory ponizej

def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0):
    """Dekorator: ponawia wywolanie przy wyjatku, z exponential backoff."""
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None

            # Probujemy kilka razy wykonac funkcje.
            for proba in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e

                    # Po ostatniej probie nie ma juz sensu czekac.
                    if proba == max_attempts - 1:
                        break

                    wait_time = delay * (backoff ** proba)
                    time.sleep(wait_time)

            raise last_error
        return wrapper
    return opakuj

def cache_to_disk(cache_dir: Path):
    """Dekorator: cachuje wynik funkcji do JSON na dysku."""
    cache_dir.mkdir(exist_ok=True, parents=True)
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Klucz robimy z nazwy funkcji, argumentow i kwargs.
            cache_key = repr((func.__name__, args, sorted(kwargs.items())))
            cache_hash = hashlib.md5(cache_key.encode("utf-8")).hexdigest()
            cache_file = cache_dir / f"{cache_hash}.json"

            # Jesli wynik juz jest na dysku, zwracamy go bez liczenia.
            if cache_file.exists():
                with cache_file.open("r", encoding="utf-8") as f:
                    return json.load(f)

            result = func(*args, **kwargs)

            # Zapisujemy wynik, zeby kolejne wywolanie bylo szybkie.
            with cache_file.open("w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            return result
        return wrapper
    return opakuj

# Czyszcze stare cache z poprzednich uruchomien, zeby eksperyment byl uczciwy.
flaky_cache_dir = WORKDIR / "flaky_cache"
flaky_cache_dir.mkdir(exist_ok=True, parents=True)
for file in flaky_cache_dir.glob("*.json"):
    file.unlink()

# Funkcja testowa z 50% szansa awarii
@cache_to_disk(flaky_cache_dir)
@retry(max_attempts=5, delay=0.05)
def flaky_fetch(text_id: int) -> dict:
    if random.random() < 0.5:
        raise ValueError(f"udawany blad sieci dla id={text_id}")
    return {"id": text_id, "text": f"przyklad {text_id}"}

# Eksperyment empiryczny
random.seed(42)
successes = 0
failures = 0

for i in range(100):
    try:
        flaky_fetch(i)
        successes += 1
    except ValueError:
        failures += 1

theoretical_success = 1 - 0.5 ** 5
experimental_success = successes / 100

print(f"Sukcesy: {successes}/100")
print(f"Porazki: {failures}/100")
print(f"P(sukces) teoretycznie: {theoretical_success:.3f}")
print(f"P(sukces) empirycznie:  {experimental_success:.3f}")

# Sprawdzam cache: drugie wywolanie tego samego id powinno isc z pliku.
first = flaky_fetch(999)
second = flaky_fetch(999)
print(f"Cache dziala: {first == second}")


Sukcesy: 96/100
Porazki: 4/100
P(sukces) teoretycznie: 0.969
P(sukces) empirycznie:  0.960
Cache dziala: True


### Wniosek do zadania 1.1

Przy jednej próbie szansa sukcesu wynosi 50%, ale przy pięciu próbach rośnie do `1 - 0.5^5 = 96.875%`. Eksperyment powinien być blisko tej wartości, a drugie wywołanie tego samego `text_id` korzysta już z cache na dysku.


---

# Lab 2 — Współbieżność i równoległość

## Teoria w trzech zdaniach

**Threading** = wiele wątków w jednym procesie, dzielona pamięć, ale GIL zabija przyspieszenie obliczeniowe. **Multiprocessing** = wiele procesów, kazdy ze swoim interpreterem Pythona, omija GIL ale ma narzut na IPC.

**Reguła kciuka:** I/O-bound (HTTP, dysk, baza) → threading. CPU-bound (parsowanie, ML, obliczenia) → multiprocessing.

**Trzecia opcja:** `asyncio` — jeden wątek, kooperatywna współbieżność. Najefektywniejsza dla I/O, ale wymaga przepisania kodu na `async`.

## Przykład rozwiązany: ThreadPool dla "I/O-bound" preprocessingu

In [4]:
from concurrent.futures import ThreadPoolExecutor
import re

# Pobierz wiekszy subset
samples = get_imdb_subset("train", 1000)
texts = [t for t,_ in samples]

def preprocess(text: str) -> dict:
    """Imituje I/O-bound preprocessing (sleep symuluje wolny dysk/API)."""
    time.sleep(0.002)  # "sieciowy" narzut
    clean = re.sub(r"<[^>]+>", " ", text).lower()
    return {"len": len(clean), "words": len(clean.split())}

# Sekwencyjnie
t0 = time.time()
seq_results = [preprocess(t) for t in texts[:200]]
seq_time = time.time() - t0
print(f"Sekwencyjnie (200 probek): {seq_time:.2f}s")

# ThreadPool
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    par_results = list(pool.map(preprocess, texts[:200]))
par_time = time.time() - t0
print(f"ThreadPool (16 workerow): {par_time:.2f}s  -- {seq_time/par_time:.1f}x szybciej")

  [timer] get_imdb_subset -> 1.21s


Sekwencyjnie (200 probek): 0.46s
ThreadPool (16 workerow): 0.07s  -- 6.5x szybciej


## Zadanie 2.1 — Multiprocessing dla CPU-bound

**Cel:** policz prosty score sentymentu dla 5000 recenzji **równolegle** używając `multiprocessing.Pool`.

**Score sentymentu (lexicon-based):**
- Lista pozytywnych słów: `["good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"]`
- Lista negatywnych słów: `["bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"]`
- Score = `(liczba pozytywnych) - (liczba negatywnych)` (case-insensitive, na pełnych słowach)

**Wymagania:**

1. Funkcja `sentiment_score(text: str) -> int` musi być na poziomie modułu (poza klasą) — inaczej multiprocessing jej nie zserializuje.
2. Porównaj **3 implementacje**: sekwencyjna, ThreadPool, multiprocessing.Pool. Wszystkie na tych samych 5000 recenzji.
3. Stwórz wykres słupkowy czasu wykonania (matplotlib).
4. **Wniosek:** który wariant najszybszy i dlaczego? (oczekiwane: multiprocessing wygrywa, bo CPU-bound i omija GIL).

**Wskazówka:** użyj `chunksize=100` w `pool.map()` żeby zmniejszyć narzut serializacji.

In [5]:
from multiprocessing import Pool
import matplotlib.pyplot as plt

POS_WORDS = {"good","great","excellent","wonderful","love","best","amazing","brilliant","perfect"}
NEG_WORDS = {"bad","worst","awful","terrible","hate","boring","waste","poor","horrible"}

def sentiment_score(text: str) -> int:
    """CPU-bound: tokenizuj, policz pozytywne minus negatywne."""
    # Zamieniam tekst na male litery i biore tylko pelne slowa.
    words = re.findall(r"\w+", text.lower())

    # Licze slowa pozytywne i negatywne.
    positive = 0
    negative = 0
    for word in words:
        if word in POS_WORDS:
            positive += 1
        if word in NEG_WORDS:
            negative += 1

    return positive - negative

# W Jupyterze na Windowsie multiprocessing najpewniej dziala,
# gdy funkcja workera jest w zwyklym pliku .py.
worker_code = """
import re

POS_WORDS = {"good","great","excellent","wonderful","love","best","amazing","brilliant","perfect"}
NEG_WORDS = {"bad","worst","awful","terrible","hate","boring","waste","poor","horrible"}

def sentiment_score_mp(text: str) -> int:
    words = re.findall(r"\\w+", text.lower())
    positive = 0
    negative = 0
    for word in words:
        if word in POS_WORDS:
            positive += 1
        if word in NEG_WORDS:
            negative += 1
    return positive - negative
"""
(WORKDIR / "sentiment_worker.py").write_text(textwrap.dedent(worker_code), encoding="utf-8")

if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))
os.environ["PYTHONPATH"] = str(WORKDIR.resolve()) + os.pathsep + os.environ.get("PYTHONPATH", "")

from sentiment_worker import sentiment_score_mp

# Pobieram te same 5000 recenzji dla wszystkich metod.
samples_sentiment = get_imdb_subset("train", 5000)
texts_5000 = [text for text, label in samples_sentiment]

# 1. Wersja sekwencyjna.
t0 = time.time()
seq_scores = [sentiment_score(text) for text in texts_5000]
seq_time = time.time() - t0

# 2. ThreadPool.
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    thread_scores = list(pool.map(sentiment_score, texts_5000))
thread_time = time.time() - t0

# 3. Multiprocessing Pool.
process_count = os.cpu_count() or 2
t0 = time.time()
with Pool(processes=process_count) as pool:
    mp_scores = pool.map(sentiment_score_mp, texts_5000, chunksize=100)
mp_time = time.time() - t0

# Prosty sanity check, czy wyniki sa takie same.
print(f"Wyniki ThreadPool takie same: {seq_scores == thread_scores}")
print(f"Wyniki multiprocessing takie same: {seq_scores == mp_scores}")

times = {
    "sekwencyjnie": seq_time,
    "ThreadPool": thread_time,
    "multiprocessing": mp_time,
}

for name, value in times.items():
    print(f"{name}: {value:.3f}s")

fastest = min(times, key=times.get)
print(f"Najszybszy wariant: {fastest}")

# Wykres slupkowy czasow.
plt.figure(figsize=(7, 4))
plt.bar(times.keys(), times.values(), color=["#4c78a8", "#f58518", "#54a24b"])
plt.ylabel("czas [s]")
plt.title("Porownanie metod liczenia sentiment_score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


  [timer] get_imdb_subset -> 0.96s


Wyniki ThreadPool takie same: True
Wyniki multiprocessing takie same: True
sekwencyjnie: 0.208s
ThreadPool: 0.323s
multiprocessing: 0.191s
Najszybszy wariant: multiprocessing


### Wniosek do zadania 2.1

To jest zadanie typu CPU-bound, więc multiprocessing może pomagać, bo omija GIL. Przy małej próbce narzut tworzenia procesów też ma znaczenie, dlatego najważniejsze jest porównanie czasów z wykresu, a nie samo założenie teoretyczne.


---

# Lab 3 — Testowanie

## Teoria w trzech zdaniach

**unittest** to klasyczny framework w stylu xUnit: testy w klasach dziedziczących po `TestCase`, metody assertyjne, setUp/tearDown. **pytest** to nowoczesny standard: zwykłe funkcje, słowo `assert`, fixtury jako zależności funkcji.

Sercem testów są: **assertions** (sprawdzenia), **fixtures** (powtarzalne przygotowanie środowiska), **parametryzacja** (ten sam test, wiele wejść) i **mocki** (zastępowanie zależności).

**Reguła:** test bez asercji to nie test. Test który zależy od kolejności uruchamiania to nie test.

## Przykład rozwiązany: unittest dla `TextStats`

In [6]:
import unittest
from io import StringIO

class TextStats:
    """Liczy proste statystyki tekstu."""
    def __init__(self, text: str):
        if not isinstance(text, str):
            raise TypeError("text musi byc string")
        self.text = text
    
    def word_count(self) -> int:
        return len(self.text.split())
    
    def char_count(self, with_spaces: bool = True) -> int:
        return len(self.text) if with_spaces else len(self.text.replace(" ", ""))
    
    def avg_word_length(self) -> float:
        words = self.text.split()
        if not words:
            return 0.0
        return sum(len(w) for w in words) / len(words)

class TestTextStats(unittest.TestCase):
    def setUp(self):
        self.empty = TextStats("")
        self.short = TextStats("Pies kot")
        self.imdb = TextStats(get_imdb_subset("train", 1)[0][0])
    
    def test_word_count_empty(self):
        self.assertEqual(self.empty.word_count(), 0)
    
    def test_word_count_short(self):
        self.assertEqual(self.short.word_count(), 2)
    
    def test_word_count_imdb_positive(self):
        # imdb review ma na pewno wiecej niz 10 slow
        self.assertGreater(self.imdb.word_count(), 10)
    
    def test_char_count_with_without_spaces(self):
        self.assertEqual(self.short.char_count(with_spaces=True), 8)
        self.assertEqual(self.short.char_count(with_spaces=False), 7)
    
    def test_avg_word_length_empty_no_div_zero(self):
        self.assertEqual(self.empty.avg_word_length(), 0.0)
    
    def test_type_check(self):
        with self.assertRaises(TypeError):
            TextStats(12345)

# Uruchom w notebooku
runner = unittest.TextTestRunner(stream=StringIO(), verbosity=2)
result = runner.run(unittest.TestLoader().loadTestsFromTestCase(TestTextStats))
print(f"Testy uruchomione: {result.testsRun}")
print(f"Sukces: {result.wasSuccessful()}")
print(f"Bledy: {len(result.errors)}, niepowodzenia: {len(result.failures)}")

  [timer] get_imdb_subset -> 0.84s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
  [timer] get_imdb_subset -> 0.00s
Testy uruchomione: 6
Sukces: True
Bledy: 0, niepowodzenia: 0


## Zadanie 3.1 — pytest dla `Tokenizer` z fixtures + parametrize

**Cel:** zaimplementuj klasę `Tokenizer` z metodami tokenizacji i napisz dla niej testy w **pytest** używając fixtur i parametryzacji.

**Specyfikacja `Tokenizer`:**

```python
class Tokenizer:
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        ...
    
    def tokenize(self, text: str) -> list[str]:
        # 1. usun tagi HTML jesli strip_html
        # 2. lowercase jesli lower
        # 3. tokeny = regex \w+ 
        # 4. odfiltruj tokeny krotsze niz min_length
        ...
    
    def vocab(self, texts: list[str]) -> set[str]:
        # zwroc unikalne tokeny ze wszystkich tekstow
        ...
```

**Wymagania testowe:**

1. **Fixture** `@pytest.fixture` o nazwie `tokenizer` zwracający `Tokenizer()` z defaultami.
2. **Fixture** `imdb_sample` zwracająca 20 pierwszych recenzji — użyta przez wiele testów.
3. **Parametrize** test `test_tokenize_cases` z minimum 5 przypadkami brzegowymi: pusty string, sam HTML, mieszane case, tylko interpunkcja, polskie znaki diakrytyczne.
4. Test który **musi zawieść** (przykładowo zły flag): oznacz `@pytest.mark.xfail`.
5. Wszystkie testy zapisz w pliku `test_tokenizer.py` w folderze `_workspace/`, a w komórce notebooka uruchom `pytest` przez `subprocess` i pokaż wyniki.

**Insight:** ile średnio unikalnych tokenów jest na 100 recenzji imdb? (heurystyka rozmiaru słownika).

In [7]:
# Zadanie 3.1 -- scaffold z konkretnymi assertami akceptacji
import subprocess

# === Krok 1: implementacja w pliku _workspace/tokenizer.py ===
tokenizer_code = """
import re

class Tokenizer:
    # Konfigurowany tokenizator: HTML strip + case + min length filter.
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        self.lower = lower
        self.strip_html = strip_html
        self.min_length = min_length

    def tokenize(self, text: str) -> list[str]:
        # Najpierw usuwam HTML, jezeli wlaczono taka opcje.
        if self.strip_html:
            text = re.sub(r"<[^>]+>", " ", text)

        # Potem robie lowercase, jezeli trzeba.
        if self.lower:
            text = text.lower()

        # Regex \\w+ lapie tez polskie litery w Pythonie 3.
        tokens = re.findall(r"\\w+", text, flags=re.UNICODE)

        # Na koncu odrzucam za krotkie tokeny.
        return [token for token in tokens if len(token) >= self.min_length]

    def vocab(self, texts: list[str]) -> set[str]:
        vocab_set = set()
        for text in texts:
            vocab_set.update(self.tokenize(text))
        return vocab_set
"""

# === Krok 2: testy z fixtures + parametrize + xfail ===
tests_code = """
import ssl
import pytest
import certifi
from tokenizer import Tokenizer

# Ten patch pomaga na Windowsie, gdy systemowe certyfikaty SSL sa popsute.
old_create_default_context = ssl.create_default_context
def create_default_context(purpose=ssl.Purpose.SERVER_AUTH, cafile=None, capath=None, cadata=None):
    if cafile is None and capath is None and cadata is None:
        cafile = certifi.where()
    return old_create_default_context(purpose=purpose, cafile=cafile, capath=capath, cadata=cadata)
ssl.create_default_context = create_default_context

@pytest.fixture
def tokenizer():
    return Tokenizer()

@pytest.fixture
def imdb_sample():
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(20))
    return [row["text"] for row in ds]

@pytest.mark.parametrize("text, expected_len", [
    ("", 0),
    ("<br><p></p>", 0),
    ("Hello WORLD!", 2),
    ("...!?!?!?", 0),
    ("zazolc gesla jazn", 3),
    ("the cat sat on the mat", 6),
])
def test_tokenize_cases(tokenizer, text, expected_len):
    assert len(tokenizer.tokenize(text)) == expected_len

def test_acceptance_examples():
    assert Tokenizer().tokenize("<br>Hello WORLD!") == ["hello", "world"]
    assert Tokenizer(lower=False).tokenize("Hello") == ["Hello"]
    assert Tokenizer(strip_html=False).tokenize("<br>hello") == ["br", "hello"]
    assert Tokenizer(min_length=4).tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]
    assert Tokenizer().vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}

def test_vocab_dedup(tokenizer):
    assert tokenizer.vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}

def test_min_length_filter():
    tok = Tokenizer(min_length=4)
    assert tok.tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]

def test_imdb_integration(tokenizer, imdb_sample):
    vocab = tokenizer.vocab(imdb_sample)
    assert len(vocab) > 500, f"za malo unikalnych tokenow: {len(vocab)}"

@pytest.mark.xfail(reason="Tokenizer nie wspiera jeszcze adresow email jako jednego tokenu")
def test_advanced_regex_unsupported():
    tok = Tokenizer()
    assert tok.tokenize("user@domain.com")[0] == "user@domain.com"
"""

# Zapisuje oba pliki do _workspace.
tokenizer_path = WORKDIR / "tokenizer.py"
tests_path = WORKDIR / "test_tokenizer.py"
tokenizer_path.write_text(textwrap.dedent(tokenizer_code), encoding="utf-8")
tests_path.write_text(textwrap.dedent(tests_code), encoding="utf-8")

# Uruchamiam pytest przez subprocess.
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(tests_path.resolve()), "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd=str(WORKDIR.resolve()),
)

print("STDOUT:")
print(result.stdout[-2000:])
if result.returncode != 0:
    print("\nSTDERR:")
    print(result.stderr[-1000:])

# Dodatkowy insight: rozmiar slownika dla 100 recenzji.
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))
from tokenizer import Tokenizer

sample_100 = get_imdb_subset("train", 100)
texts_100 = [text for text, label in sample_100]
tok = Tokenizer()
unique_100 = len(tok.vocab(texts_100))
print(f"Unikalne tokeny w 100 recenzjach: {unique_100}")
print(f"Srednio na 1 recenzje: {unique_100 / 100:.1f}")


STDOUT:
============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\jakub\miniconda3\envs\lab_aap\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\jakub\Documents\Codex\2026-06-26\s\outputs\zestaw_zaliczeniowy\_workspace
plugins: anyio-4.13.0
collecting ... collected 11 items

test_tokenizer.py::test_tokenize_cases[-0] PASSED                        [  9%]
test_tokenizer.py::test_tokenize_cases[<br><p></p>-0] PASSED             [ 18%]
test_tokenizer.py::test_tokenize_cases[Hello WORLD!-2] PASSED            [ 27%]
test_tokenizer.py::test_tokenize_cases[...!?!?!?-0] PASSED               [ 36%]
test_tokenizer.py::test_tokenize_cases[zazolc gesla jazn-3] PASSED       [ 45%]
test_tokenizer.py::test_tokenize_cases[the cat sat on the mat-6] PASSED  [ 54%]
test_tokenizer.py::test_acceptance_examples PASSED                       [ 63%]
test_tokenizer.py::test_vocab_dedup PASSED                       

  [timer] get_imdb_subset -> 0.84s
Unikalne tokeny w 100 recenzjach: 5053
Srednio na 1 recenzje: 50.5


### Wniosek do zadania 3.1

Testy `pytest` dobrze pasują do tokenizatora, bo łatwo sprawdzają wiele przypadków przez `parametrize`. Rozmiar słownika dla 100 recenzji pokazuje, że nawet mały fragment IMDb daje dużo unikalnych tokenów.


---

# Lab 4 — Bazy danych

## Teoria w trzech zdaniach

**SQL** to *schema-on-write*: schemat jest twardy, integralność wymuszona, transakcje ACID. **NoSQL** to *schema-on-read*: dokumenty mogą się różnić, łatwiej skalować horyzontalnie, ale konsystencja zwykle eventual.

**Złota zasada:** wybierasz bazę pod **wzorzec zapytań**, nie pod "jakie mam dane". Jeśli czytasz/piszesz całe dokumenty — NoSQL. Jeśli robisz joiny i agregacje na wymiarach — SQL.

**W SQLite od Pythona 3.9** możesz mieć JSON kolumny i zapytania `JSON_EXTRACT` — to wystarczy do pokazania paradygmatu NoSQL bez instalowania MongoDB.

## Przykład rozwiązany: imdb → SQLite + analityka

In [8]:
import sqlite3

DB_PATH = WORKDIR / "imdb.db"
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# Schemat -- klasyczna relacja
cur.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

# Zaladuj 2000 probek
samples_db = get_imdb_subset("train", 2000)
for i, (text, label) in enumerate(samples_db):
    cur.execute(
        "INSERT INTO reviews (id, text, label, word_count, char_count) VALUES (?, ?, ?, ?, ?)",
        (i, text, label, len(text.split()), len(text))
    )
conn.commit()

# Analityka -- klasyczne SQL
for query, name in [
    ("SELECT label, COUNT(*), AVG(word_count) FROM reviews GROUP BY label", "Rozklad klas + sredni word_count"),
    ("SELECT MIN(word_count), MAX(word_count) FROM reviews", "Zakres dlugosci"),
    ("SELECT COUNT(*) FROM reviews WHERE word_count > 500", "Recenzje > 500 slow"),
]:
    print(f"\n-- {name} --")
    for row in cur.execute(query):
        print(f"  {row}")
conn.close()

  [timer] get_imdb_subset -> 0.89s

-- Rozklad klas + sredni word_count --
  (0, 1000, 224.705)
  (1, 1000, 232.164)

-- Zakres dlugosci --
  (12, 1005)

-- Recenzje > 500 slow --
  (164,)


## Zadanie 4.1 — NoSQL-style w SQLite (JSON column)

**Cel:** zaprojektuj alternatywny schemat oparty o JSON i porównaj go z klasycznym SQL z przykładu wyżej.

**Wymagania:**

1. Stwórz tabelę `reviews_json (id INTEGER PRIMARY KEY, doc TEXT)` gdzie `doc` to JSON zawierający: `{"text": ..., "label": ..., "stats": {"word_count": ..., "sentiment_hint": "pos"|"neg"}, "tags": [...]}`.
2. Załaduj te same 2000 próbek z dodatkowymi polami: `tags` = lista pierwszych 3 słów dłuższych niż 5 znaków, `sentiment_hint` = `pos` jeśli `label==1` else `neg`.
3. Napisz 4 zapytania w stylu NoSQL używając `json_extract(doc, '$.path')`:
   - Rozkład klas (count per `sentiment_hint`).
   - Średni `word_count` dla każdej klasy.
   - Recenzje gdzie `tags` zawiera słowo "movie" (`LIKE '%movie%'` na JSON).
   - Top 5 najdłuższych recenzji w klasie pozytywnej.
4. **Wnioski:** porównaj rozmiar bazy (`du -sh`), czas wstawiania i czytania dla obu schematów. Który schemat jest lepszy dla *tego* problemu i dlaczego?

In [9]:
# Zadanie 4.1 -- NoSQL style w SQLite (scaffold)

DB_JSON = WORKDIR / "imdb_json.db"
if DB_JSON.exists():
    DB_JSON.unlink()

conn2 = sqlite3.connect(str(DB_JSON))
cur2 = conn2.cursor()

# Krok 1: schemat z JSON column
cur2.execute("""
CREATE TABLE reviews_json (
    id INTEGER PRIMARY KEY,
    doc TEXT NOT NULL
)
""")

# Krok 2: zaladuj te same 2000 probek jako JSON dokumenty
samples_nosql = get_imdb_subset("train", 2000)

insert_start = time.time()
for i, (text, label) in enumerate(samples_nosql):
    # Proste czyszczenie slow pod tagi.
    words = re.findall(r"\w+", text.lower())
    tags = []
    for word in words:
        if len(word) > 5:
            tags.append(word)
        if len(tags) == 3:
            break

    doc = {
        "text": text,
        "label": label,
        "stats": {
            "word_count": len(text.split()),
            "sentiment_hint": "pos" if label == 1 else "neg",
        },
        "tags": tags,
    }
    cur2.execute(
        "INSERT INTO reviews_json (id, doc) VALUES (?, ?)",
        (i, json.dumps(doc, ensure_ascii=False)),
    )
conn2.commit()
json_insert_time = time.time() - insert_start

# Krok 3: cztery zapytania w stylu NoSQL z json_extract
queries = {
    "rozklad_klas": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint, COUNT(*) AS n
        FROM reviews_json
        GROUP BY hint
    """,
    "avg_word_count_per_class": """
        SELECT
            json_extract(doc, '$.stats.sentiment_hint') AS hint,
            ROUND(AVG(json_extract(doc, '$.stats.word_count')), 1) AS avg_words
        FROM reviews_json
        GROUP BY hint
    """,
    "tags_zawiera_movie": """
        SELECT
            id,
            json_extract(doc, '$.stats.sentiment_hint') AS hint,
            json_extract(doc, '$.tags') AS tags
        FROM reviews_json
        WHERE json_extract(doc, '$.tags') LIKE '%movie%'
        LIMIT 10
    """,
    "top5_najdluzsze_pozytywne": """
        SELECT
            id,
            json_extract(doc, '$.stats.word_count') AS word_count,
            substr(json_extract(doc, '$.text'), 1, 80) AS preview
        FROM reviews_json
        WHERE json_extract(doc, '$.label') = 1
        ORDER BY json_extract(doc, '$.stats.word_count') DESC
        LIMIT 5
    """,
}

json_read_start = time.time()
for name, sql in queries.items():
    print(f"\n-- {name} --")
    for row in cur2.execute(sql):
        print(f"  {row}")
json_read_time = time.time() - json_read_start

# Krok 4: porownanie rozmiaru i czasu
# Dla uczciwego porownania robie mala kopie SQL i mierze insert.
DB_SQL_COMPARE = WORKDIR / "imdb_sql_compare.db"
if DB_SQL_COMPARE.exists():
    DB_SQL_COMPARE.unlink()

conn_sql = sqlite3.connect(str(DB_SQL_COMPARE))
cur_sql = conn_sql.cursor()
cur_sql.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

sql_insert_start = time.time()
for i, (text, label) in enumerate(samples_nosql):
    cur_sql.execute(
        "INSERT INTO reviews (id, text, label, word_count, char_count) VALUES (?, ?, ?, ?, ?)",
        (i, text, label, len(text.split()), len(text)),
    )
conn_sql.commit()
sql_insert_time = time.time() - sql_insert_start

sql_read_start = time.time()
sql_rows = list(cur_sql.execute("SELECT label, COUNT(*), AVG(word_count) FROM reviews GROUP BY label"))
sql_read_time = time.time() - sql_read_start

import os as _os
size_sql = _os.path.getsize(DB_SQL_COMPARE)
size_json = _os.path.getsize(DB_JSON)

print(f"\n=== Porownanie ===")
print(f"SQL schema (reviews):       {size_sql:>9,} bajtow")
print(f"JSON schema (reviews_json): {size_json:>9,} bajtow")
print(f"SQL insert:  {sql_insert_time:.4f}s")
print(f"JSON insert: {json_insert_time:.4f}s")
print(f"SQL read:    {sql_read_time:.4f}s -> {sql_rows}")
print(f"JSON read:   {json_read_time:.4f}s")

conn_sql.close()
conn2.close()


  [timer] get_imdb_subset -> 0.00s

-- rozklad_klas --
  ('neg', 1000)
  ('pos', 1000)

-- avg_word_count_per_class --
  ('neg', 224.7)
  ('pos', 232.2)

-- tags_zawiera_movie --
  (50, 'pos', '["totally","movies","journey"]')
  (61, 'neg', '["movies","before","sittings"]')
  (64, 'pos', '["gangster","movies","should"]')
  (122, 'neg', '["movies","premiere","extremely"]')
  (137, 'neg', '["considered","myself","movies"]')
  (147, 'neg', '["possibly","movies","critics"]')
  (150, 'neg', '["movies","should","seemed"]')
  (193, 'pos', '["famous","movies","french"]')
  (268, 'pos', '["favourite","movies","dialogue"]')
  (281, 'neg', '["please","without","movies"]')

-- top5_najdluzsze_pozytywne --
  (1109, 982, 'The interesting aspect of "The Apprentice" is it demonstrates that the tradition')
  (1557, 973, 'Yesterday, I went to the monthly Antique Flea Market that comes to town. I reall')
  (1526, 969, '******WARNING: MAY CONTAIN SPOILERS**************<br /><br />So who are these "M')
  (

### Wniosek do zadania 4.1

Dla tego problemu klasyczny SQL jest wygodniejszy do agregacji i zwykle mniejszy na dysku. JSON column jest bardziej elastyczny, ale zapytania są mniej czytelne i przy analizie trzeba częściej używać `json_extract`.


---

# Lab 5 — PySpark

## Teoria w trzech zdaniach

**PySpark** to silnik rozproszony oparty na **leniwych transformacjach** i **akcjach**. Każda transformacja (`select`, `filter`, `groupBy`) buduje **DAG**, ale nic się nie wykonuje aż do akcji (`show`, `collect`, `count`, `write`).

**Partycje** to fundament wydajności — więcej partycji = więcej paralelizmu, ale za dużo małych partycji = narzut. Reguła kciuka: 2-4 partycje na rdzeń CPU.

**Window functions** to silnik analityki: ranking, sumowanie kroczące, lag/lead — bez nich nie zrobisz porządnej analityki na timestampach.

## Przykład rozwiązany: imdb → Spark + count słów per klasa

In [10]:
from pyspark.sql import SparkSession, functions as F

# Setup Sparka -- robust
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

# Szukam Javy: najpierw ustawiona zmienna, potem typowe miejsca.
java_candidates = [
    os.environ.get("JAVA_HOME"),
    str(Path(sys.prefix) / "Library"),
    str(Path(sys.prefix) / "Library" / "jre"),
    r"C:\Program Files\Eclipse Adoptium\jdk-17",
    r"C:\Program Files\Java\jdk-17",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home",
    "/usr/lib/jvm/java-17-openjdk-amd64",
]

for candidate in java_candidates:
    if candidate and os.path.exists(Path(candidate) / "bin" / "java.exe"):
        os.environ["JAVA_HOME"] = candidate
        break
    if candidate and os.path.exists(Path(candidate) / "bin" / "java"):
        os.environ["JAVA_HOME"] = candidate
        break

spark = (SparkSession.builder
    .appName("AAP zaliczenie")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', 'brak')}")

# Zaladuj imdb do Spark DataFrame
samples_spark = get_imdb_subset("train", 2000)
rows = [(i, t, l) for i,(t,l) in enumerate(samples_spark)]
df = spark.createDataFrame(rows, ["id", "text", "label"])

# Liczba slow per klasa
df_words = (df
    .withColumn("words", F.split(F.lower(F.regexp_replace("text", r"<[^>]+>", " ")), r"\W+"))
    .withColumn("word_count", F.size("words")))

print("\n-- Statystyki per klasa --")
df_words.groupBy("label").agg(
    F.count("*").alias("n"),
    F.round(F.avg("word_count"), 1).alias("avg_words"),
    F.expr("percentile_approx(word_count, 0.5)").alias("median_words")
).show()

# Najczestsze slowa per klasa (top 10 pozytywne)
df_exploded = df_words.select("label", F.explode("words").alias("word"))
df_exploded = df_exploded.filter((F.length("word") > 3) & (F.col("label") == 1))
print("\n-- Top 10 slow w pozytywnych recenzjach --")
df_exploded.groupBy("word").count().orderBy(F.col("count").desc()).limit(10).show()


Spark 4.1.2 ready
JAVA_HOME: C:\Users\jakub\miniconda3\envs\lab_aap\Library
  [timer] get_imdb_subset -> 0.00s



-- Statystyki per klasa --


+-----+----+---------+------------+
|label|   n|avg_words|median_words|
+-----+----+---------+------------+
|    1|1000|    237.1|         173|
|    0|1000|    230.5|         178|
+-----+----+---------+------------+


-- Top 10 slow w pozytywnych recenzjach --


+-----+-----+
| word|count|
+-----+-----+
| this| 2821|
| that| 2800|
| with| 1837|
| film| 1802|
|movie| 1527|
| have| 1013|
| from|  803|
| they|  794|
| like|  718|
| very|  658|
+-----+-----+



## Zadanie 5.1 — Window functions: ranking recenzji

**Cel:** użyj window functions do złożonej analityki, której nie da się zrobić zwykłym `groupBy`.

**Wymagania:**

1. Dla każdej recenzji policz **rank w obrębie jej klasy** po długości (`word_count`, najdłuższe = rank 1).
2. Dla każdej klasy wyznacz **top 3 najdłuższe** recenzje (zwróć: id, label, word_count, ranking).
3. Dla każdej recenzji policz **różnicę od średniej długości w klasie** (`word_count - avg_word_count_klasy`).
4. **Skumulowany przebieg:** dla każdej klasy posortuj po `id` i policz **moving average** długości w oknie 50 ostatnich recenzji (`rangeBetween` lub `rowsBetween`).
5. Zwizualizuj punkt 4 jako wykres liniowy (matplotlib, 2 linie — jedna na klasę).

**Wskazówka:** użyj `pyspark.sql.Window`:

```python
from pyspark.sql.window import Window
w = Window.partitionBy("label").orderBy(F.col("word_count").desc())
df.withColumn("rank", F.row_number().over(w))
```

In [11]:
# TODO Zadanie 5.1: window functions
from pyspark.sql.window import Window

# 1. Ranking recenzji w obrebie klasy po dlugosci.
rank_window = Window.partitionBy("label").orderBy(F.col("word_count").desc(), F.col("id").asc())

# 3. Srednia dlugosc w klasie.
avg_window = Window.partitionBy("label")

# 4. Moving average z 50 ostatnich recenzji w ramach klasy.
moving_window = Window.partitionBy("label").orderBy("id").rowsBetween(-49, 0)

df_ranked = (df_words
    .withColumn("ranking", F.row_number().over(rank_window))
    .withColumn("avg_word_count_klasy", F.avg("word_count").over(avg_window))
    .withColumn("diff_from_class_avg", F.col("word_count") - F.col("avg_word_count_klasy"))
    .withColumn("moving_avg_50", F.avg("word_count").over(moving_window)))

# 2. Top 3 najdluzsze recenzje w kazdej klasie.
print("-- Top 3 najdluzsze recenzje per klasa --")
df_ranked.filter(F.col("ranking") <= 3).select(
    "id", "label", "word_count", "ranking"
).orderBy("label", "ranking").show()

print("-- Przyklad roznicy od sredniej klasowej --")
df_ranked.select(
    "id",
    "label",
    "word_count",
    F.round("avg_word_count_klasy", 1).alias("avg_class"),
    F.round("diff_from_class_avg", 1).alias("diff"),
).orderBy("id").show(10)

# Dane do wykresu przenosze do pandas, bo mamy tylko 2000 wierszy.
plot_pd = (df_ranked
    .select("id", "label", "moving_avg_50")
    .orderBy("label", "id")
    .toPandas())

plt.figure(figsize=(9, 4))
for label in sorted(plot_pd["label"].unique()):
    part = plot_pd[plot_pd["label"] == label]
    plt.plot(part["id"], part["moving_avg_50"], label=f"label={label}")

plt.title("Moving average word_count, okno 50 recenzji")
plt.xlabel("id")
plt.ylabel("srednia liczba slow")
plt.legend()
plt.tight_layout()
plt.show()


-- Top 3 najdluzsze recenzje per klasa --


+----+-----+----------+-------+
|  id|label|word_count|ranking|
+----+-----+----------+-------+
| 518|    0|      1020|      1|
|1869|    0|      1018|      2|
| 566|    0|      1014|      3|
|1109|    1|      1000|      1|
|1526|    1|       998|      2|
|1557|    1|       996|      3|
+----+-----+----------+-------+

-- Przyklad roznicy od sredniej klasowej --


+---+-----+----------+---------+------+
| id|label|word_count|avg_class|  diff|
+---+-----+----------+---------+------+
|  0|    1|       127|    237.1|-110.1|
|  1|    1|       133|    237.1|-104.1|
|  2|    0|       185|    230.5| -45.5|
|  3|    1|       123|    237.1|-114.1|
|  4|    0|       664|    230.5| 433.5|
|  5|    1|       208|    237.1| -29.1|
|  6|    1|       120|    237.1|-117.1|
|  7|    0|       171|    230.5| -59.5|
|  8|    0|       313|    230.5|  82.5|
|  9|    1|       252|    237.1|  14.9|
+---+-----+----------+---------+------+
only showing top 10 rows


### Wniosek do zadania 5.1

Window functions pozwalają liczyć wartości zależne od grupy bez zwijania danych do jednego wiersza. Dzięki temu dla każdej recenzji zostaje `id`, `label`, ranking, różnica od średniej i moving average.


---

# Lab 6 — Data Quality (jakość danych)

## Teoria w trzech zdaniach

**Data Quality** to nie audyt po wszystkim — to **kontrakt** który dane muszą spełnić zanim wejdą do pipeline'u. Sześć wymiarów: kompletność, unikalność, poprawność, zgodność, świeżość, integralność.

Współczesny stack: `pandera`/`great_expectations` dla **deklaratywnych testów**, `pandas-profiling` (teraz `ydata-profiling`) dla **raportów eksploracyjnych**, własne **walidatory** dla reguł biznesowych.

**Reguła:** jeśli nie potrafisz w jednym zdaniu opisać co znaczy "dobre dane" dla Twojego problemu, nie powinieneś jeszcze trenować modelu.

## Przykład rozwiązany: profilowanie imdb — wykrywanie anomalii

In [12]:
import pandas as pd

# Wczytaj wieksza probke
samples_dq = get_imdb_subset("train", 2000)
df_pd = pd.DataFrame(samples_dq, columns=["text", "label"])
df_pd["word_count"] = df_pd["text"].str.split().str.len()
df_pd["char_count"] = df_pd["text"].str.len()

# Profil podstawowy
print("=== KOMPLETNOSC ===")
nulls = df_pd.isnull().sum()
print(f"Nulle: {dict(nulls)}")

print("\n=== UNIKALNOSC ===")
dup_count = df_pd["text"].duplicated().sum()
print(f"Duplikaty tekstu: {dup_count}")

print("\n=== ROZKLAD LABELI ===")
print(df_pd["label"].value_counts(normalize=True).rename("frac"))
balance_ratio = df_pd["label"].value_counts().min() / df_pd["label"].value_counts().max()
print(f"Stosunek mniejszosci do wiekszosci: {balance_ratio:.3f} (1.0 = idealnie zbalansowane)")

print("\n=== ANOMALIE DLUGOSCI ===")
p99 = df_pd["word_count"].quantile(0.99)
p01 = df_pd["word_count"].quantile(0.01)
outliers = df_pd[(df_pd["word_count"] > p99) | (df_pd["word_count"] < p01)]
print(f"P1: {p01:.0f}, P99: {p99:.0f}, outlierow (poza P1-P99): {len(outliers)}")

print("\n=== ANOMALIE TRESCI ===")
has_html = df_pd["text"].str.contains(r"<[^>]+>", regex=True).sum()
very_short = (df_pd["word_count"] < 5).sum()
print(f"Tekst zawiera HTML tagi: {has_html} ({has_html/len(df_pd)*100:.1f}%)")
print(f"Bardzo krotkie recenzje (<5 slow): {very_short}")

print("\nINSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!")

  [timer] get_imdb_subset -> 0.00s
=== KOMPLETNOSC ===
Nulle: {'text': np.int64(0), 'label': np.int64(0), 'word_count': np.int64(0), 'char_count': np.int64(0)}

=== UNIKALNOSC ===
Duplikaty tekstu: 0

=== ROZKLAD LABELI ===
label
1    0.5
0    0.5
Name: frac, dtype: float64
Stosunek mniejszosci do wiekszosci: 1.000 (1.0 = idealnie zbalansowane)

=== ANOMALIE DLUGOSCI ===
P1: 42, P99: 889, outlierow (poza P1-P99): 39

=== ANOMALIE TRESCI ===
Tekst zawiera HTML tagi: 1189 (59.5%)
Bardzo krotkie recenzje (<5 slow): 0

INSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!


## Zadanie 6.1 — Kontrakt danych + raport JSON

**Cel:** zaimplementuj prosty *Data Quality Framework* w czystym Pythonie i wygeneruj raport o jakości datasetu.

**Wymagania:**

1. Klasa `DataContract` z metodą `add_rule(name, callable, severity)` (severity ∈ {`info`, `warning`, `error`}).
2. Klasa `DataValidator` która iteruje po regułach kontraktu i zwraca raport: `{rule_name: {passed: bool, severity, details}}`.
3. Zdefiniuj kontrakt dla imdb z **minimum 6 regułami**:
   - `no_nulls` — brak NULL w `text` i `label`
   - `labels_in_set` — wszystkie labele są w {0, 1}
   - `min_word_count` — każda recenzja ma min. 5 słów
   - `max_word_count` — żadna recenzja > 2000 słów (sanity)
   - `no_duplicates` — brak duplikatów `text`
   - `class_balance` — stosunek klas między 0.5 a 1.5
4. Reguły o severity `error` które zawiodły powinny rzucić wyjątek (fail fast). Reszta jest tylko ostrzeżeniem.
5. Wygeneruj raport w pliku `_workspace/data_quality_report.json` z timestampem.

**Bonus:** zaimplementuj `severity="warning"` regułę `no_html_tags` i pokaż że *raport* o niej mówi, ale walidacja nie zawodzi.

In [13]:
# TODO Zadanie 6.1: DataContract + DataValidator
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Rule:
    name: str
    check: Callable
    severity: str = "warning"  # info | warning | error

class DataContract:
    def __init__(self, name: str):
        self.name = name
        self.rules: list[Rule] = []

    def add_rule(self, name: str, check: Callable, severity: str = "warning"):
        # Sprawdzam, czy severity jest z dozwolonej listy.
        if severity not in {"info", "warning", "error"}:
            raise ValueError("severity musi byc: info, warning albo error")

        self.rules.append(Rule(name=name, check=check, severity=severity))
        return self

class DataValidator:
    def __init__(self, contract: DataContract):
        self.contract = contract

    def validate(self, df) -> dict:
        report = {}
        failed_errors = []

        # Uruchamiam kazda regule po kolei.
        for rule in self.contract.rules:
            try:
                passed, details = rule.check(df)
                passed = bool(passed)
            except Exception as e:
                passed = False
                details = f"blad reguly: {e}"

            report[rule.name] = {
                "passed": passed,
                "severity": rule.severity,
                "details": details,
            }

            if rule.severity == "error" and not passed:
                failed_errors.append(rule.name)

        if failed_errors:
            raise ValueError(f"Nieprzeszly reguly error: {failed_errors}")

        return report

# Proste funkcje walidujace dane.
def check_no_nulls(df):
    nulls = int(df[["text", "label"]].isnull().sum().sum())
    return nulls == 0, f"liczba nulli w text/label: {nulls}"

def check_labels_in_set(df):
    labels = set(df["label"].dropna().unique())
    return labels.issubset({0, 1}), f"etykiety w danych: {sorted(labels)}"

def check_min_word_count(df):
    bad_count = int((df["word_count"] < 5).sum())
    return bad_count == 0, f"recenzje krotsze niz 5 slow: {bad_count}"

def check_max_word_count(df):
    too_long = int((df["word_count"] > 2000).sum())
    max_words = int(df["word_count"].max())
    return too_long == 0, f"recenzje > 2000 slow: {too_long}, max={max_words}"

def check_no_duplicates(df):
    duplicates = int(df["text"].duplicated().sum())
    return duplicates == 0, f"duplikaty text: {duplicates}"

def check_class_balance(df):
    counts = df["label"].value_counts()
    ratio = counts.max() / counts.min()
    return 0.5 <= ratio <= 1.5, f"licznosci={dict(counts)}, ratio={ratio:.3f}"

def check_no_html_tags(df):
    html_count = int(df["text"].str.contains(r"<[^>]+>", regex=True).sum())
    return html_count == 0, f"teksty z tagami HTML: {html_count}"

# Zbuduj kontrakt dla imdb -- minimum 6 regul
contract = DataContract("imdb_contract")
contract.add_rule("no_nulls", check_no_nulls, severity="error")
contract.add_rule("labels_in_set", check_labels_in_set, severity="error")
contract.add_rule("min_word_count", check_min_word_count, severity="error")
contract.add_rule("max_word_count", check_max_word_count, severity="warning")
contract.add_rule("no_duplicates", check_no_duplicates, severity="error")
contract.add_rule("class_balance", check_class_balance, severity="error")
contract.add_rule("no_html_tags", check_no_html_tags, severity="warning")

# Uruchom walidacje
validator = DataValidator(contract)
rules_report = validator.validate(df_pd)

full_report = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "contract": contract.name,
    "rules": rules_report,
}

# Zapisz raport do JSON
report_path = WORKDIR / "data_quality_report.json"
report_path.write_text(json.dumps(full_report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Zapisano raport: {report_path.resolve()}")
for name, result in rules_report.items():
    status = "OK" if result["passed"] else "NIE OK"
    print(f"{name:18s} {status:6s} [{result['severity']}] - {result['details']}")


Zapisano raport: C:\Users\jakub\Documents\Codex\2026-06-26\s\outputs\zestaw_zaliczeniowy\_workspace\data_quality_report.json
no_nulls           OK     [error] - liczba nulli w text/label: 0
labels_in_set      OK     [error] - etykiety w danych: [np.int64(0), np.int64(1)]
min_word_count     OK     [error] - recenzje krotsze niz 5 slow: 0
max_word_count     OK     [warning] - recenzje > 2000 slow: 0, max=1005
no_duplicates      OK     [error] - duplikaty text: 0
class_balance      OK     [error] - licznosci={1: np.int64(1000), 0: np.int64(1000)}, ratio=1.000
no_html_tags       NIE OK [warning] - teksty z tagami HTML: 1189


### Wniosek do zadania 6.1

Reguły `error` pilnują warunków, bez których pipeline nie powinien iść dalej. Reguła `no_html_tags` jest ostrzeżeniem, bo HTML w IMDb jest realnym problemem jakości, ale nie blokuje wykonania notebooka.


---

# Sekcja kontrolna — co umiesz po tym zestawie

Po ukończeniu wszystkich 6 zadań powinieneś **bez przygotowania** odpowiedzieć na:

1. **Dekorator:** kiedy `functools.wraps` jest konieczny, a kiedy można sobie odpuścić?
2. **Concurrency:** dlaczego threading nie przyspieszy obliczeń, a multiprocessing przyspieszy?
3. **Testowanie:** kiedy lepiej parametrize, a kiedy osobne testy?
4. **Bazy:** co znaczy *schema-on-read*? Daj praktyczny przykład gdy to plus, a kiedy minus.
5. **Spark:** co to znaczy że transformacja jest "lazy"? Daj przykład **kiedy to boli** w debugowaniu.
6. **Data Quality:** różnica między *audytem* a *kontraktem* danych. W produkcji potrzebujesz obu — dlaczego?

## Co dalej?

- **Pakowanie:** `pyproject.toml`, `poetry`, dystrybucja przez `pip`
- **CI/CD:** GitHub Actions, pre-commit hooks, automated testing
- **Observability:** logging structured (`structlog`), metryki (`prometheus_client`), tracing (`opentelemetry`)
- **Orchestration:** Apache Airflow / Prefect / Dagster do *prawdziwych* pipeline'ów
- **Workshops:** [Real Python](https://realpython.com), [Talk Python To Me](https://talkpython.fm) podcast

In [14]:
# Sprzatanie
try:
    spark.stop()
    print("Spark zatrzymany.")
except NameError:
    pass
print(f"Workspace: {WORKDIR.resolve()}")
print("Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.")

Spark zatrzymany.
Workspace: C:\Users\jakub\Documents\Codex\2026-06-26\s\outputs\zestaw_zaliczeniowy\_workspace
Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.
